# ABSA Comment Labeler — UF HiPerGator · NaviGator API

**Model:** `Llama-3.3-70B-Instruct`  
**Task:** Aspect-Based Sentiment Analysis + Category Classification (~189k comments)  
**Output:** `data/comments_labeled.csv` + `data/comments_labeled.jsonl`

## 1. Imports & Logging

In [ ]:
import csv
import json
import logging
import random
import sys
import time
from datetime import datetime
from pathlib import Path
import os
from dotenv import load_dotenv
import re as _re

try:
    import requests
except ImportError:
    raise ImportError("'requests' not found. Run: pip install requests --user  then restart kernel.")

# ── Logging: prints to notebook output AND writes to a file ──
log = logging.getLogger("absa")
log.setLevel(logging.INFO)
log.handlers.clear()  # avoid duplicate handlers on re-run

_fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
_sh  = logging.StreamHandler(sys.stdout)
_sh.setFormatter(_fmt)
_fh  = logging.FileHandler("absa_labeler.log", mode="a")
_fh.setFormatter(_fmt)
log.addHandler(_sh)
log.addHandler(_fh)

log.info("Imports OK — %s", datetime.now().isoformat())

2026-03-23 14:25:49,055 [INFO] Imports OK — 2026-03-23T14:25:49.055144


## 2. Configuration

In [ ]:
# ── API Keys (4 keys for ~189k rows) ──
load_dotenv(".env")
API_KEYS = os.environ.get("NAVIGATOR_API_KEYS", "").split(",")
API_KEYS = [k.strip() for k in API_KEYS if k.strip()]

# ── Paths ──
INPUT_CSV    = "../data/comments_for_analysis.csv"
OUTPUT_CSV   = "../data/comments_labeled.csv"
OUTPUT_JSONL = "../data/comments_labeled.jsonl"
COST_REPORT  = "../data/cost_report.json"

# ── Resume mode ──
# True  → skip comment_ids already present in OUTPUT_CSV and append new results.
# False → overwrite output files and start from scratch.
RESUME = True

# ── Model / API ──
NAVIGATOR_BASE_URL = "https://api.ai.it.ufl.edu"
MODEL_NAME         = "llama-3.3-70b-instruct"
REQUEST_TIMEOUT    = 45    # seconds per call
MAX_RETRIES        = 3
RETRY_BASE_DELAY   = 1.0  # seconds — exponential backoff base

# ── Cost control ──
COST_PER_M_IN   = 0.40   # USD per million input  tokens
COST_PER_M_OUT  = 0.40   # USD per million output tokens
BUDGET_ALERT_AT = 24.50  # USD — retire a key when it hits this threshold

# ── Progress reporting ──
CHECKPOINT_EVERY = 200   # flush to disk + print cost table every N rows

# ── Smoke-test (set to None to run the full dataset) ──
LIMIT = 20400  # e.g. LIMIT = 20  to label only the first 20 rows

# ── Validation sets ──
VALID_CATEGORIES = {"AI_Voice_Related", "Business_Model_Related",
                    "Game_Related", "Others"}
VALID_SENTIMENTS = {"Pos", "Neg", "Neu", "Ambivalent"}

log.info("Config loaded — %d API key(s) | input: %s", len(API_KEYS), INPUT_CSV)

2026-03-24 14:07:55,449 [INFO] Config loaded — 4 API key(s) | input: data/comments_for_analysis.csv


## 3. System Prompt

In [ ]:
SYSTEM_PROMPT = """You are an expert Aspect-Based Sentiment Analysis (ABSA) annotator.
Task:
Using the PRIMARY topic of the comment, assign ONE category, ONE sentiment, and ONE confidence score.

Inputs:
video_title
video_context
parent_context (context only)
comment (target)

Return ONLY valid JSON:

{
"category":"AI_Voice_Related|Business_Model_Related|Game_Related|Others",
"sentiment":"Pos|Neg|Neu|Ambivalent",
"confidence":"0.00–1.00",
"reasoning":"60-100 word explanation"
}

Category rules:

AI_Voice_Related → AI voices, TTS, voice acting replacement, ethics  
Business_Model_Related → pricing, monetisation, subscriptions, ads  
Game_Related → gameplay, graphics, bugs, updates, design, experience,
or general reactions to how the game looks (fun, hype, interesting, amazing).
Others → unrelated discussion  

Hierarchy (apply in order):

1 identify primary topic (strongest opinion)
2 if multiple topics exist → choose topic with strongest sentiment evidence
3 AI voice → AI_Voice_Related
4 monetisation → Business_Model_Related
5 game experience → Game_Related
6 otherwise → Others

Sentiment:
- Pos = praise
- Neg = criticism
- Neu = factual/question
- Ambivalent = both

- criticism → Neg
- praise → Pos
- both → Ambivalent
- none → Neu
- sarcasm → implied meaning

Edge:
- bugs → Game + Neg
- pricing complaints → Business + Neg
- AI questions → AI + Neu

Evidence:
- MUST cite exact words from comment
- Use context only if needed
- Do NOT invent evidence

Confidence (text clarity):
- 0.90+ explicit topic + sentiment
- 0.75+ clear
- 0.60+ weak
- 0.40 ambiguity
- <0.40 multiple interpretations
- Lower for sarcasm or mixed signals
- Format: two decimals

Reasoning must:
- identify main aspect
- cite sentiment words
- justify category choice
- justify confidence

Do NOT:
- add fields or multiple labels
- infer demographics
- rely on title alone
- default Others without reason

Return JSON only."""

print("System prompt set. Length:", len(SYSTEM_PROMPT), "chars")

System prompt set. Length: 1898 chars


## 4. Key Rotator & Cost Tracker Classes

In [23]:
class KeyRotator:
    """
    Round-robin key rotation.
    Retired keys (budget exhausted / auth error) are removed from the pool.
    """
    def __init__(self, keys):
        if not keys:
            raise ValueError("Provide at least one API key.")
        self._keys   = list(keys)
        self._active = list(range(len(keys)))  # indices of live keys
        self._idx    = 0

    @property
    def available_count(self):
        return len(self._active)

    def current(self):
        if not self._active:
            raise RuntimeError("All API keys have been retired — no budget remaining.")
        return self._keys[self._active[self._idx % len(self._active)]]

    def advance(self):
        """Step to the next live key after a successful request."""
        if self._active:
            self._idx = (self._idx + 1) % len(self._active)

    def retire(self, key):
        """Permanently remove a key from rotation."""
        try:
            ki = self._keys.index(key)
        except ValueError:
            return
        if ki in self._active:
            self._active.remove(ki)
            log.warning("Key …%s RETIRED. Keys remaining: %d", key[-6:], len(self._active))
            if self._active:
                self._idx = self._idx % len(self._active)


class CostTracker:
    """
    Tracks token usage and USD cost per API key.
    Flags a key as over-budget once it exceeds BUDGET_ALERT_AT.
    """
    def __init__(self, keys, threshold=BUDGET_ALERT_AT):
        self._threshold = threshold
        self._data = {
            k: {"tokens_in": 0, "tokens_out": 0, "requests": 0, "cost_usd": 0.0}
            for k in keys
        }

    def record(self, key, tokens_in, tokens_out):
        """Add usage for one request. Returns the key's cumulative cost."""
        d = self._data[key]
        d["tokens_in"]  += tokens_in
        d["tokens_out"] += tokens_out
        d["requests"]   += 1
        cost = (tokens_in  / 1_000_000 * COST_PER_M_IN +
                tokens_out / 1_000_000 * COST_PER_M_OUT)
        d["cost_usd"] += cost
        return d["cost_usd"]

    def cost(self, key):
        return self._data[key]["cost_usd"]

    def over_budget(self, key):
        return self._data[key]["cost_usd"] >= self._threshold

    def total_cost(self):
        return sum(d["cost_usd"] for d in self._data.values())

    def print_summary(self):
        print("\n" + "=" * 62)
        print(f"  {'Key (last 6)':>14}  {'Requests':>9}  {'Tokens In':>11}  {'Tokens Out':>11}  {'Cost USD':>9}")
        print("  " + "-" * 60)
        for k, d in self._data.items():
            print(f"  …{k[-6:]:>13}  {d['requests']:>9,}  {d['tokens_in']:>11,}  {d['tokens_out']:>11,}  ${d['cost_usd']:>8.4f}")
        print("  " + "-" * 60)
        print(f"  {'TOTAL':>14}  {'':>9}  {'':>11}  {'':>11}  ${self.total_cost():>8.4f}")
        print("=" * 62 + "\n")

    def save(self, path=COST_REPORT):
        with open(path, "w") as f:
            json.dump(self._data, f, indent=2)
        log.info("Cost report saved → %s", path)


print("KeyRotator and CostTracker defined.")

KeyRotator and CostTracker defined.


## 5. Helper Functions (Prompt Builder, API Caller, Parser)

In [ ]:
def build_user_message(row):
    """Construct the user-turn message from one CSV row."""
    title   = (row.get("video_title")    or "").strip()[:300]
    context = (row.get("video_context")  or "").strip()[:400]
    parent  = (row.get("parent_context") or "").strip()[:300]
    comment = (row.get("text_preprocessed") or "").strip()

    parts = [
        f"video_title: {title}",
        f"video_context: {context}",
    ]
    if parent:   # only include parent_context when it exists
        parts.append(f"parent_context: {parent}")
    parts.append(f"comment: {comment}")
    return "\n".join(parts)


# ── API caller ──
def call_navigator(key, user_message):
    """
    Send one chat-completion request to the NaviGator API.
    Returns the full JSON response dict.
    Raises requests.HTTPError on 4xx/5xx.
    """
    url     = f"{NAVIGATOR_BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type":  "application/json",}
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},],
        "temperature": 0.0,
        "max_tokens":  512,
        "response_format": {"type": "json_object"},}  # enforce JSON mode
    resp = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return resp.json()

# ── Parse Helpers ──
def _clean_raw(raw):
    """Strip markdown fences."""
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return raw.strip()

def _repair_json(raw):
    """
    Replace literal newlines/carriage-returns inside JSON string values.
    This is the most common cause of 'Expecting , delimiter' errors when
    the model writes a multi-line reasoning value without escaping it.
    """
    def _fix_value(m):
        inner = m.group(2).replace("\n", " ").replace("\r", "")
        return m.group(1) + inner + m.group(3)
    return _re.sub(
        r'("(?:category|sentiment|confidence|reasoning)"\s*:\s*")(.*?)("\s*[,}])',
        _fix_value, raw, flags=_re.DOTALL,
    )

def _regex_extract(raw):
    """Last-resort field extraction when JSON parsing fails completely."""
    cat_m  = _re.search(r'"category"\s*:\s*"([^"]+)"', raw)
    sent_m = _re.search(r'"sentiment"\s*:\s*"([^"]+)"', raw)
    # Greedy dotall so embedded newlines in reasoning are captured
    reas_m = (_re.search(r'"reasoning"\s*:\s*"(.*?)"\s*[,}]', raw, _re.DOTALL)
              or _re.search(r'"reasoning"\s*:\s*"(.+)', raw, _re.DOTALL))
    if not (cat_m and sent_m):
        return None
    reasoning = (reas_m.group(1).strip().rstrip('"') if reas_m else "")
    return {"category": cat_m.group(1), "sentiment": sent_m.group(1), "reasoning": reasoning}

def _normalise(label):
    """Guard against completely unknown values; no alias remapping."""
    cat  = label["category"]
    sent = label["sentiment"]
    if cat  not in VALID_CATEGORIES: cat  = "Others"
    if sent not in VALID_SENTIMENTS: sent = "Neu"
    return cat, sent


# ── Response parser ──
def _clean_raw(raw):
    """Strip markdown fences."""
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return raw.strip()

def _repair_json(raw):
    """
    Replace literal newlines/carriage-returns inside JSON string values.
    This is the most common cause of 'Expecting , delimiter' errors when
    the model writes a multi-line reasoning value without escaping them.
    """
    def _fix_value(m):
        inner = m.group(2).replace("\n", " ").replace("\r", "")
        return m.group(1) + inner + m.group(3)
    return _re.sub(
        r'("(?:category|sentiment|confidence|reasoning)"\s*:\s*")(.*?)("\s*[,}])',
        _fix_value, raw, flags=_re.DOTALL,
    )

def _regex_extract(raw):
    """Last-resort field extraction when JSON parsing fails completely."""
    cat_m  = _re.search(r'"category"\s*:\s*"([^"]+)"', raw)
    sent_m = _re.search(r'"sentiment"\s*:\s*"([^"]+)"', raw)
    conf_m = _re.search(r'"confidence"\s*:\s*"([^"]+)"', raw)
    reas_m = (_re.search(r'"reasoning"\s*:\s*"(.*?)"\s*[,}]', raw, _re.DOTALL)
              or _re.search(r'"reasoning"\s*:\s*"(.+)', raw, _re.DOTALL))
    if not (cat_m and sent_m):
        return None
    reasoning  = (reas_m.group(1).strip().rstrip('"') if reas_m else "")
    confidence = (conf_m.group(1).strip() if conf_m else "")
    return {
        "category":   cat_m.group(1),
        "sentiment":  sent_m.group(1),
        "confidence": confidence,
        "reasoning":  reasoning,
    }

def _normalise(label):
    """Guard against unknown values; no alias remapping."""
    cat  = label["category"]
    sent = label["sentiment"]
    if cat  not in VALID_CATEGORIES: cat  = "Others"
    if sent not in VALID_SENTIMENTS: sent = "Neu"
    return cat, sent


# ── Main response parser ──
def parse_response(api_response):
    """
    3-strategy parser — in order:
      1. json.loads on cleaned raw          (fast, strict)
      2. json.loads after newline repair    (fixes unescaped line breaks)
      3. regex field extraction             (nuclear option)
    Always returns a dict so the loop never crashes.
    """
    raw = ""
    try:
        raw = api_response["choices"][0]["message"]["content"].strip()
    except (KeyError, IndexError) as exc:
        return {"category": "PARSE_ERROR", "sentiment": "PARSE_ERROR",
                "confidence": "", "reasoning": f"Bad API response structure: {exc}",
                "raw_text": raw, "parse_ok": False}

    cleaned = _clean_raw(raw)
    label   = None

    # Strategy 1 — strict json.loads
    try:
        label = json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Strategy 2 — repair newlines then retry
    if label is None:
        try:
            label = json.loads(_repair_json(cleaned))
            log.info("Repaired JSON parsed OK for snippet: %s", cleaned[:60])
        except json.JSONDecodeError:
            pass

    # Strategy 3 — regex extraction
    if label is None:
        label = _regex_extract(cleaned)
        if not label:
            log.warning("All parse strategies failed. raw: %s", raw[:200])
            return {"category": "PARSE_ERROR", "sentiment": "PARSE_ERROR",
                    "confidence": "", "reasoning": f"All strategies failed. raw: {raw[:300]}",
                    "raw_text": raw, "parse_ok": False}

    category, sentiment = _normalise(label)
    confidence = label.get("confidence", "")
    reasoning  = label.get("reasoning", "").replace("\n", " ").strip()

    return {"category": category, "sentiment": sentiment, "confidence": confidence,
            "reasoning": reasoning, "raw_text": cleaned, "parse_ok": True}


# ── Token extractor ──
def extract_usage(api_response):
    """Return (prompt_tokens, completion_tokens) from the usage field."""
    u = api_response.get("usage", {})
    return u.get("prompt_tokens", 0), u.get("completion_tokens", 0)


print("Helper functions defined.")

Helper functions defined.


## 6. I/O Helpers (CSV Load / Write, JSONL, Resume)

In [26]:
OUTPUT_FIELDS = [
    "comment_id", "video_title", "video_context", "parent_context",
    "text_preprocessed", "category", "sentiment", "confidence", "reasoning",
    "api_key_used", "tokens_in", "tokens_out", "parse_ok",]


def load_csv(path):
    """Load CSV into a list of dicts. Tries utf-8-sig first, then latin-1."""
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            with open(path, newline="", encoding=enc) as f:
                rows = list(csv.DictReader(f))
            log.info("Loaded %d rows from %s  (encoding=%s)", len(rows), path, enc)
            return rows
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Cannot decode: {path}")


def load_done_ids(output_path):
    """Return the set of comment_ids already written to output_path."""
    done = set()
    if not Path(output_path).exists():
        return done
    try:
        with open(output_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                cid = row.get("comment_id", "").strip()
                if cid:
                    done.add(cid)
        log.info("Resume: %d comment_ids already labeled in %s", len(done), output_path)
    except Exception as exc:
        log.warning("Could not read existing output (%s) — starting fresh.", exc)
    return done


def open_writers(output_csv, output_jsonl, resume):
    """Open both output files. Append mode when resuming, overwrite otherwise."""
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    Path(output_jsonl).parent.mkdir(parents=True, exist_ok=True)

    csv_mode  = "a" if (resume and Path(output_csv).exists())   else "w"
    jsonl_mode = "a" if (resume and Path(output_jsonl).exists()) else "w"

    csv_fh    = open(output_csv,   csv_mode,  newline="", encoding="utf-8")
    writer    = csv.DictWriter(csv_fh, fieldnames=OUTPUT_FIELDS, extrasaction="ignore")
    if csv_mode == "w":
        writer.writeheader()

    jsonl_fh  = open(output_jsonl, jsonl_mode, encoding="utf-8")

    return csv_fh, writer, jsonl_fh


print("I/O helpers defined.")

I/O helpers defined.


## 7. Load & Validate Dataset

In [27]:
all_rows = load_csv(INPUT_CSV)

# ── Validate required columns ──
REQUIRED_COLS = {"comment_id", "video_title", "video_context", "text_preprocessed"}
if all_rows:
    missing = REQUIRED_COLS - set(all_rows[0].keys())
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}")
    print(f"Columns OK. Found columns: {list(all_rows[0].keys())}")

# ── Row limit for testing ──
if LIMIT:
    all_rows = all_rows[:LIMIT]
    log.info("LIMIT=%d applied — processing only %d rows.", LIMIT, LIMIT)

print(f"\nTotal rows to consider: {len(all_rows):,}")

2026-03-24 14:08:01,456 [INFO] Loaded 189726 rows from data/comments_for_analysis.csv  (encoding=utf-8-sig)
Columns OK. Found columns: ['comment_id', 'parent_id', 'parent_text', 'text_preprocessed', 'video_title', 'video_context', 'video_id', 'comment_date', 'author_hash', 'days_from_genai_announcement', 'days_from_business_model_change', 'days_from_game_announcement', 'days_from_early_access', 'days_from_game_launch', 'mentions_genai', 'mentions_business_model', 'mentions_game', 'likes', 'like_count_log', 'is_spike', 'z_score']
2026-03-24 14:08:01,656 [INFO] LIMIT=20400 applied — processing only 20400 rows.

Total rows to consider: 20,400


## 8. Main Processing Loop

> **This is the long-running cell.** Progress is printed every 200 rows.  
> Safe to interrupt (`Kernel → Interrupt`) and re-run — `RESUME = True` will skip already-labeled rows.

In [ ]:
# ── Initialise state ──
rotator = KeyRotator(API_KEYS)
tracker = CostTracker(API_KEYS)

done_ids = load_done_ids(OUTPUT_CSV) if RESUME else set()
pending  = [r for r in all_rows if str(r.get("comment_id", "")).strip() not in done_ids]

log.info("Total: %d | Already done: %d | Pending: %d",
         len(all_rows), len(done_ids), len(pending))

if not pending:
    print("Nothing to do — all comments are already labeled. ✓")
else:
    csv_fh, csv_writer, jsonl_fh = open_writers(OUTPUT_CSV, OUTPUT_JSONL, RESUME)
    processed = 0
    errors    = 0
    start_t   = time.time()

    try:
        for i, row in enumerate(pending):
            comment_id = str(row.get("comment_id", f"row_{i}")).strip()
            user_msg   = build_user_message(row)

            # ── Retry loop ──
            label    = None
            t_in = t_out = 0
            key_used = None
            last_exc = None

            for attempt in range(1, MAX_RETRIES + 1):
                if rotator.available_count == 0:
                    log.error("All API keys exhausted — stopping.")
                    raise RuntimeError("No API keys remaining.")

                key = rotator.current()

                try:
                    api_resp = call_navigator(key, user_msg)
                    t_in, t_out = extract_usage(api_resp)
                    label    = parse_response(api_resp)
                    key_used = key

                    # Record cost; retire key if over budget
                    cumulative_cost = tracker.record(key, t_in, t_out)
                    if tracker.over_budget(key):
                        log.warning(
                            "Key …%s reached $%.4f — retiring (threshold $%.2f).",
                            key[-6:], cumulative_cost, BUDGET_ALERT_AT
                        )
                        rotator.retire(key)
                    else:
                        rotator.advance()   # step to next key on success

                    last_exc = None
                    break  # ← exit retry loop on success

                except requests.exceptions.HTTPError as exc:
                    status   = exc.response.status_code if exc.response is not None else 0
                    last_exc = exc
                    if status == 429:            # rate-limited → rotate immediately
                        log.warning("[%d] 429 on key …%s (attempt %d) — rotating key.",
                                    i + 1, key[-6:], attempt)
                        rotator.advance()
                        time.sleep(RETRY_BASE_DELAY * attempt)
                    elif status in (401, 403):   # auth error → retire key
                        log.error("Auth error (HTTP %d) on key …%s — retiring.",
                                  status, key[-6:])
                        rotator.retire(key)
                    else:
                        log.warning("[%d] HTTP %d (attempt %d) — retrying.",
                                    i + 1, status, attempt)
                        time.sleep(RETRY_BASE_DELAY ** attempt)

                except (requests.exceptions.ConnectionError,
                        requests.exceptions.Timeout) as exc:
                    last_exc = exc
                    delay = RETRY_BASE_DELAY ** attempt + random.uniform(0, 1)
                    log.warning("[%d] Network error (attempt %d): %s — wait %.1fs",
                                i + 1, attempt, exc, delay)
                    time.sleep(delay)

                except Exception as exc:
                    last_exc = exc
                    log.error("Unexpected error (attempt %d): %s", attempt, exc)
                    time.sleep(RETRY_BASE_DELAY)

            # ── Build output row ──
            if label is None:
                errors += 1
                log.error("FAILED comment_id=%s after %d attempts: %s",
                          comment_id, MAX_RETRIES, last_exc)
                label = {
                    "category":  "ERROR",
                    "confidence": "ERROR", 
                    "sentiment": "ERROR",
                    "reasoning": f"API call failed: {last_exc}",
                    "parse_ok":  False,
                }

            out_row = {
                "comment_id":        comment_id,
                "video_title":       row.get("video_title",        ""),
                "video_context":     row.get("video_context",      ""),
                "parent_context":    row.get("parent_context",     ""),
                "text_preprocessed": row.get("text_preprocessed",  ""),
                "category":          label["category"],
                "confidence":        label["confidence"],
                "sentiment":         label["sentiment"],
                "reasoning":         label["reasoning"],
                "api_key_used":      f"…{key_used[-6:]}" if key_used else "FAILED",
                "tokens_in":         t_in,
                "tokens_out":        t_out,
                "parse_ok":          label.get("parse_ok", False),
            }

            csv_writer.writerow(out_row)
            jsonl_fh.write(json.dumps(out_row, ensure_ascii=False) + "\n")
            processed += 1

            # ── Periodic checkpoint ──
            if processed % CHECKPOINT_EVERY == 0:
                csv_fh.flush()
                jsonl_fh.flush()
                tracker.save()

                elapsed = time.time() - start_t
                rate    = processed / elapsed if elapsed > 0 else 0
                eta_s   = (len(pending) - processed) / rate if rate > 0 else 0
                pct     = 100 * processed / len(pending)

                print(f"\n[{datetime.now().strftime('%H:%M:%S')}] "
                      f"{processed:,}/{len(pending):,} ({pct:.1f}%) | "
                      f"{rate:.2f} rows/s | ETA {eta_s/60:.0f} min | "
                      f"Errors: {errors}")
                tracker.print_summary()

    except KeyboardInterrupt:
        log.warning("Interrupted by user after %d rows. Outputs saved up to this point.",
                    processed)
    finally:
        csv_fh.flush();   csv_fh.close()
        jsonl_fh.flush(); jsonl_fh.close()
        tracker.save()

        elapsed = time.time() - start_t
        print("\n" + "=" * 62)
        print(f"  DONE — {processed:,} rows in {elapsed/60:.1f} min | "
              f"Errors: {errors}")
        tracker.print_summary()
        log.info("Finished. processed=%d errors=%d elapsed=%.1f min",
                 processed, errors, elapsed / 60)

2026-03-24 14:08:01,890 [INFO] Resume: 20200 comment_ids already labeled in data/comments_labeled.csv
2026-03-24 14:08:01,904 [INFO] Total: 20400 | Already done: 20200 | Pending: 200
2026-03-24 14:20:34,697 [INFO] Cost report saved → data/cost_report.json

[14:20:34] 200/200 (100.0%) | 0.27 rows/s | ETA 0 min | Errors: 0

    Key (last 6)   Requests    Tokens In   Tokens Out   Cost USD
  ------------------------------------------------------------
  …       InyxBg         50       27,758        5,932  $  0.0135
  …       bceQ1A         50       28,063        5,990  $  0.0136
  …       RLxqww         50       28,015        6,001  $  0.0136
  …       xT2YjA         50       28,352        6,197  $  0.0138
  ------------------------------------------------------------
           TOTAL                                       $  0.0545

2026-03-24 14:20:34,700 [INFO] Cost report saved → data/cost_report.json

  DONE — 200 rows in 12.5 min | Errors: 0

    Key (last 6)   Requests    Tokens In  

## 9. Final Cost Report & Quick Sanity Check

In [29]:
# ── Print final cost summary ──
print("Final cost breakdown:")
tracker.print_summary()

# ── Quick sanity check on the output CSV ──
if Path(OUTPUT_CSV).exists():
    labeled_rows = load_csv(OUTPUT_CSV)
    total_labeled = len(labeled_rows)

    parse_errors = sum(1 for r in labeled_rows if r.get("parse_ok") == "False")
    api_errors   = sum(1 for r in labeled_rows if r.get("category") == "ERROR")

    from collections import Counter
    cat_counts  = Counter(r["category"]  for r in labeled_rows)
    sent_counts = Counter(r["sentiment"] for r in labeled_rows)

    print(f"\nOutput file : {OUTPUT_CSV}")
    print(f"Total rows  : {total_labeled:,}")
    print(f"Parse errors: {parse_errors}")
    print(f"API errors  : {api_errors}")
    print(f"\nCategory distribution:")
    for cat, n in cat_counts.most_common():
        print(f"  {cat:<30} {n:>7,}  ({100*n/total_labeled:.1f}%)")
    print(f"\nSentiment distribution:")
    for sent, n in sent_counts.most_common():
        print(f"  {sent:<20} {n:>7,}  ({100*n/total_labeled:.1f}%)")
else:
    print(f"Output file not found: {OUTPUT_CSV}")

Final cost breakdown:

    Key (last 6)   Requests    Tokens In   Tokens Out   Cost USD
  ------------------------------------------------------------
  …       InyxBg         50       27,758        5,932  $  0.0135
  …       bceQ1A         50       28,063        5,990  $  0.0136
  …       RLxqww         50       28,015        6,001  $  0.0136
  …       xT2YjA         50       28,352        6,197  $  0.0138
  ------------------------------------------------------------
           TOTAL                                       $  0.0545

2026-03-24 14:20:46,350 [INFO] Loaded 20400 rows from data/comments_labeled.csv  (encoding=utf-8-sig)

Output file : data/comments_labeled.csv
Total rows  : 20,400
Parse errors: 0
API errors  : 0

Category distribution:
  Game_Related                    16,275  (79.8%)
  Others                           2,742  (13.4%)
  Business_Model_Related           1,274  (6.2%)
  AI_Voice_Related                   109  (0.5%)

Sentiment distribution:
  Neg            

### 10. Re-run Only Failed Rows (Optional)

Run this cell **after** the main loop finishes if there are still rows with `category == ERROR` or `parse_ok == False` that we need to retry.

In [ ]:
# ── Filter failed rows from the labeled output ──
if Path(OUTPUT_CSV).exists():
    labeled_rows = load_csv(OUTPUT_CSV)
    failed_ids   = {
        r["comment_id"]
        for r in labeled_rows
        if r.get("category") in ("ERROR", "PARSE_ERROR") or r.get("parse_ok") == "False"
    }
    print(f"Failed / parse-error rows: {len(failed_ids)}")

    if failed_ids:
        # Remove the failed rows from the output so RESUME will re-process them
        good_rows = [r for r in labeled_rows if r["comment_id"] not in failed_ids]
        tmp_path  = OUTPUT_CSV + ".tmp"

        with open(tmp_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(good_rows)

        import os
        os.replace(tmp_path, OUTPUT_CSV)
        print(f"Removed {len(failed_ids)} failed rows. Rewrite output CSV with {len(good_rows):,} clean rows.")
        print("Now re-run Cell 8 (RESUME=True) to retry only the failed comments.")
    else:
        print("No failed rows — nothing to retry. ✓")
else:
    print(f"Output not found: {OUTPUT_CSV}")